[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ItsNotAILABS/PARALLAX-Exchange-Clearinghouse/blob/main/examples/01_place_order.ipynb)

# 01 — Place a Simulated Trade on PARALLAX Exchange

This notebook demonstrates how the **Phantom Exchange** matching engine works:
- Create trading pairs
- Place limit orders (buy & sell)
- Watch the matching engine execute trades at **zero gas fees**
- Inspect the resulting order book

In [ ]:
# Setup — runs in Google Colab with zero install
!pip install -q numpy pandas

import time
from dataclasses import dataclass, field
from typing import List, Dict
from enum import Enum

# PHI constants
PHI = 1.6180339887498948482
PHI_INV = 1.0 / PHI
PHI_INV_3 = PHI_INV ** 3
HEARTBEAT_MS = (PHI ** 4) * (1000.0 / 7.83)

print(f"φ = {PHI:.10f}")
print(f"Heartbeat = {HEARTBEAT_MS:.1f}ms")
print("✅ Ready")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHANTOM EXCHANGE — Order Book & Matching Engine
# ═══════════════════════════════════════════════════════════════

class OrderSide(Enum):
    BUY = "buy"
    SELL = "sell"

class OrderType(Enum):
    LIMIT = "limit"
    MARKET = "market"

class OrderStatus(Enum):
    OPEN = "open"
    FILLED = "filled"
    PARTIALLY_FILLED = "partially_filled"
    CANCELLED = "cancelled"

@dataclass
class Order:
    order_id: int
    pair_id: str
    trader: str
    side: OrderSide
    order_type: OrderType
    price: float
    quantity: float
    filled_quantity: float = 0.0
    status: OrderStatus = OrderStatus.OPEN
    timestamp: float = field(default_factory=time.time)

@dataclass
class Fill:
    fill_id: int
    pair_id: str
    buyer: str
    seller: str
    price: float
    quantity: float
    timestamp: float
    gas_fee: float = 0.0  # ALWAYS zero

class PhantomExchange:
    def __init__(self):
        self.order_id_counter = 0
        self.fill_id_counter = 0
        self.orders: Dict[str, List[Order]] = {}
        self.fills: List[Fill] = []
        self.pairs: Dict[str, dict] = {}
    
    def create_pair(self, pair_id: str, base: str, quote: str):
        self.pairs[pair_id] = {
            "base": base, "quote": quote,
            "tick_size": PHI_INV_3,
            "last_price": 0.0, "volume_24h": 0.0
        }
        self.orders[pair_id] = []
    
    def place_order(self, pair_id: str, trader: str, side: OrderSide,
                    order_type: OrderType, price: float, quantity: float) -> Order:
        self.order_id_counter += 1
        order = Order(
            order_id=self.order_id_counter, pair_id=pair_id,
            trader=trader, side=side, order_type=order_type,
            price=price, quantity=quantity
        )
        self.orders[pair_id].append(order)
        self._match(pair_id)
        return order
    
    def _match(self, pair_id: str):
        orders = self.orders[pair_id]
        buys = sorted([o for o in orders if o.side == OrderSide.BUY and o.status == OrderStatus.OPEN],
                      key=lambda o: (-o.price, o.timestamp))
        sells = sorted([o for o in orders if o.side == OrderSide.SELL and o.status == OrderStatus.OPEN],
                       key=lambda o: (o.price, o.timestamp))
        for buy in buys:
            for sell in sells:
                if buy.status != OrderStatus.OPEN or sell.status != OrderStatus.OPEN:
                    continue
                if buy.price >= sell.price:
                    fill_qty = min(buy.quantity - buy.filled_quantity,
                                   sell.quantity - sell.filled_quantity)
                    fill_price = sell.price
                    self.fill_id_counter += 1
                    self.fills.append(Fill(
                        fill_id=self.fill_id_counter, pair_id=pair_id,
                        buyer=buy.trader, seller=sell.trader,
                        price=fill_price, quantity=fill_qty,
                        timestamp=time.time()
                    ))
                    buy.filled_quantity += fill_qty
                    sell.filled_quantity += fill_qty
                    buy.status = OrderStatus.FILLED if buy.filled_quantity >= buy.quantity else OrderStatus.PARTIALLY_FILLED
                    sell.status = OrderStatus.FILLED if sell.filled_quantity >= sell.quantity else OrderStatus.PARTIALLY_FILLED
                    self.pairs[pair_id]["last_price"] = fill_price
                    self.pairs[pair_id]["volume_24h"] += fill_qty * fill_price
    
    def get_order_book(self, pair_id: str) -> dict:
        orders = self.orders.get(pair_id, [])
        bids = sorted([(o.price, o.quantity - o.filled_quantity)
                for o in orders if o.side == OrderSide.BUY and o.status == OrderStatus.OPEN], reverse=True)
        asks = sorted([(o.price, o.quantity - o.filled_quantity)
                for o in orders if o.side == OrderSide.SELL and o.status == OrderStatus.OPEN])
        return {"bids": bids, "asks": asks}

print("✅ PhantomExchange class defined")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 1: Create a trading pair
# ═══════════════════════════════════════════════════════════════

exchange = PhantomExchange()
exchange.create_pair("ICP_USDT", "ICP", "USDT")

print("Trading Pair Created:")
print(f"  Pair: ICP/USDT")
print(f"  Tick Size: {exchange.pairs['ICP_USDT']['tick_size']:.6f} (φ⁻³ derived)")
print(f"  Gas Fee: $0.00 (organism pays all costs)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 2: Place sell orders (market makers provide liquidity)
# ═══════════════════════════════════════════════════════════════

# Market makers place sell orders
sell_1 = exchange.place_order("ICP_USDT", "maker_alice", OrderSide.SELL, OrderType.LIMIT, 12.50, 100.0)
sell_2 = exchange.place_order("ICP_USDT", "maker_bob", OrderSide.SELL, OrderType.LIMIT, 12.45, 50.0)
sell_3 = exchange.place_order("ICP_USDT", "maker_carol", OrderSide.SELL, OrderType.LIMIT, 12.55, 200.0)

# Market makers place buy orders
buy_1 = exchange.place_order("ICP_USDT", "maker_dave", OrderSide.BUY, OrderType.LIMIT, 12.30, 75.0)
buy_2 = exchange.place_order("ICP_USDT", "maker_eve", OrderSide.BUY, OrderType.LIMIT, 12.25, 150.0)

print("Order Book (before taker):")
book = exchange.get_order_book("ICP_USDT")
print(f"\n  {'ASKS (sells)':>30}")
for price, qty in reversed(book['asks']):
    print(f"  {'':>20} {qty:>8.1f} @ ${price:.2f}")
print(f"  {'--- spread ---':>30}")
for price, qty in book['bids']:
    print(f"  ${price:.2f} × {qty:<8.1f}")
print(f"  {'BIDS (buys)':>30}")
print(f"\n  Spread: ${book['asks'][0][0] - book['bids'][0][0]:.2f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 3: Taker places a buy order that crosses the spread
# ═══════════════════════════════════════════════════════════════

print("🔥 TAKER ACTION: Buy 80 ICP @ limit $12.50")
print("=" * 50)

taker_order = exchange.place_order(
    "ICP_USDT", "taker_frank", OrderSide.BUY, OrderType.LIMIT, 12.50, 80.0
)

print(f"\nOrder Status: {taker_order.status.value}")
print(f"Filled: {taker_order.filled_quantity:.1f} / {taker_order.quantity:.1f} ICP")

print(f"\n{'─' * 50}")
print("FILLS EXECUTED:")
print(f"{'─' * 50}")
for fill in exchange.fills:
    print(f"  Fill #{fill.fill_id}:")
    print(f"    {fill.buyer} ← {fill.quantity:.1f} ICP @ ${fill.price:.2f} ← {fill.seller}")
    print(f"    Total cost: ${fill.quantity * fill.price:.2f}")
    print(f"    Gas fee: ${fill.gas_fee:.2f} ✨ ZERO")
    print()

print(f"Last Price: ${exchange.pairs['ICP_USDT']['last_price']:.2f}")
print(f"24h Volume: ${exchange.pairs['ICP_USDT']['volume_24h']:.2f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4: View final order book state
# ═══════════════════════════════════════════════════════════════

book = exchange.get_order_book("ICP_USDT")

print("Final Order Book:")
print(f"\n  {'ASKS':>30}")
for price, qty in reversed(book['asks']):
    print(f"  {'':>20} {qty:>8.1f} @ ${price:.2f}")
if not book['asks']:
    print(f"  {'':>20} (empty)")
print(f"  {'--- spread ---':>30}")
for price, qty in book['bids']:
    print(f"  ${price:.2f} × {qty:<8.1f}")
if not book['bids']:
    print(f"  (empty)")
print(f"  {'BIDS':>30}")

print(f"\n✅ Trade complete — settled in {HEARTBEAT_MS:.0f}ms with ZERO gas fees")